In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI


In [2]:
load_dotenv(override=True)

True

In [3]:
api_key = os.getenv("OPENAI_API_KEY")

In [4]:
if api_key is None:
    raise ValueError("OPENAI_API_KEY is not set in the environment variables")
else:
    print("OPENAI_API_KEY is set in the environment variables")

client = OpenAI(api_key=api_key)

OPENAI_API_KEY is set in the environment variables


In [6]:
messages = [{'role': 'user', 'content': 'Hello, how are you?'}]
messages


[{'role': 'user', 'content': 'Hello, how are you?'}]

In [13]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
#     temperature=0.7,
#     max_tokens=100,
#     top_p=1,
)

# response.choices[1].message.content
# print(dict(response))
print(response.__dict__.keys())

dict_keys(['id', 'choices', 'created', 'model', 'object', 'service_tier', 'system_fingerprint', 'usage', '_request_id'])


In [14]:
print(response.choices[0].message.__dict__.keys())

dict_keys(['content', 'refusal', 'role', 'annotations', 'audio', 'function_call', 'tool_calls'])


In [15]:
import httpx
from bs4 import BeautifulSoup

def fetch_website_content(url):
    try:
        response = httpx.get(url)
        response.raise_for_status()
        return response.text
    except Exception as e:
        print(f"Error fetching content from {url}: {e}")
        return None 
    

In [20]:
from IPython.display import Image, display,Markdown

# Markdown(fetch_website_content("https://edwarddonner.com/"))

In [31]:
import httpx
from bs4 import BeautifulSoup
from IPython.display import Markdown

def fetch_website_info(url):
    try:
        response = httpx.get(url, follow_redirects=True)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.text, 'html.parser')
        
        for element in soup(["script", "style", "header", "footer", "nav"]):
            element.decompose()

        return soup.get_text(separator="\n", strip=True)

    except Exception as e:
        return f"Error: {e}"

# content = fetch_website_info("https://edwarddonner.com/")
# Markdown(content)

In [32]:
System_prompt = """You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

In [ ]:
User_prompt = f"""Here are the contents of a website.
{website}
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

In [34]:
messages = [{"role": "system", "content": System_prompt}, {"role": "user", "content": User_prompt}]

In [35]:
def messages_for(website):
    return [{"role": "system", "content": System_prompt}, {"role": "user", "content": User_prompt.format(website=website)}]


In [36]:
def summarize_website(website):
    website_content = fetch_website_info(website)
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages_for(website_content),
        # temperature=0.7,
        # max_tokens=100,
    )
    return response.choices[0].message.content


In [37]:
summarize_website("https://edwarddonner.com/")

"Sure! Just drop the contents of the website here, and I'll throw some snarky magic on it!"